In [1]:
import pandas as pd
import networkx as nx
import ast

In [2]:
data_root = "/home/chris/school/csc502/CSC-502-PSCAN/data/output/pscan_output/"
datasets = [
    "lfr_5000",
    #"lfr_10000",
    #"lfr_20000",
    #"lfr_40000",
    #"lfr_80000",
    #"lfr_160000"
    ]
epsilons = ["0.2", "0.4", "0.6", "0.8", "1.0", ]

In [ ]:
results = []
for dataset in datasets:
    print(dataset)
    adj_file = data_root+"adjlists/"+dataset+".adjlist"
    G = nx.read_adjlist(adj_file)
    result = {"dataset": dataset}
    for epsilon in epsilons:

        cluster_file = data_root+"clusters/"+dataset+f".sim_eps{epsilon}_clusters.csv"
        clusters_df = pd.read_csv(cluster_file, header=None, names=["cluster_id", "nodes"])
        clusters_df["nodes"] = clusters_df["nodes"].apply(ast.literal_eval)

        node_to_cluster = {}

        for _, row in clusters_df.iterrows():
            cluster_id = int(row["cluster_id"])
            for node in row["nodes"]:
                node_to_cluster[str(node)] = cluster_id   # graph nodes are strings
        print(len(node_to_cluster))
        print(len(G.nodes))

        for i in G.nodes():
            if i in node_to_cluster:
                G.nodes[i]["cluster"] = int(node_to_cluster[i])
            else:
                G.nodes[i]["cluster"] = -1
        
        communities = {}
        for node, data in G.nodes(data=True):
            cluster_id = data.get("cluster", -1)
            if cluster_id not in communities:
                communities[cluster_id] = set()
            communities[cluster_id].add(node)
        communities_list = list(communities.values())

        modularity = nx.community.modularity(G, communities_list)
        result[f"eps{epsilon}"] = modularity
    results.append(result)

df = pd.DataFrame(results)
df.set_index('dataset')

In [ ]:
df

In [ ]:
node_to_cluster